<a href="https://colab.research.google.com/github/PRIYA-P14/Hospital_Resource_Allocation_Analysis/blob/main/Hospital_Resource_Allocation_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load the Dataset

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
file_path = "/content/drive/MyDrive/Hospital_Resource_Allocation/Dataset/hospital_resource_allocation_mock_150.csv"

In [4]:
import pandas as pd

df = pd.read_csv(file_path)

df.head()

,Month,Department,Patient_Volume,Average_Treatment_Duration,Staff_Count,Equipment_Utilisation
0,Jan,Cardiology,834,47.7,18,92.7
1,Jan,Neurology,598,52.7,16,73.6
2,Jan,Orthopedics,748,44.5,20,69.2
3,Jan,Pediatrics,695,28.8,22,59.0
4,Jan,Emergency,1051,23.9,24,97.9


In [5]:
df.shape

(150, 6)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Month                       150 non-null    object 
 1   Department                  150 non-null    object 
 2   Patient_Volume              150 non-null    int64  
 3   Average_Treatment_Duration  150 non-null    float64
 4   Staff_Count                 150 non-null    int64  
 5   Equipment_Utilisation       150 non-null    float64
dtypes: float64(2), int64(2), object(2)
memory usage: 7.2+ KB


In [7]:
df.describe()

,Patient_Volume,Average_Treatment_Duration,Staff_Count,Equipment_Utilisation
count,150.000000,150.000000,150.000000,150.000000
mean,672.200000,41.922667,17.726667,72.751333
std,220.821043,13.571690,3.979641,12.713691
min,279.000000,19.900000,11.000000,42.100000
25%,509.250000,29.525000,15.000000,65.550000
50%,618.000000,42.150000,16.000000,72.000000
75%,819.250000,48.800000,21.000000,78.025000
max,1289.000000,79.400000,26.000000,98.000000


In [8]:
df.head()
df.tail()

,Month,Department,Patient_Volume,Average_Treatment_Duration,Staff_Count,Equipment_Utilisation
145,Dec,Dermatology,429,29.5,11,58.6
146,Dec,Gynecology,728,44.5,18,74.1
147,Dec,ENT,510,28.6,14,57.9
148,Dec,Gastroenterology,608,48.0,15,75.2
149,Dec,Pulmonology,676,46.6,16,76.7


Data cleaning and validation


In [9]:
df.isnull().sum()

,0
Month,0
Department,0
Patient_Volume,0
Average_Treatment_Duration,0
Staff_Count,0
Equipment_Utilisation,0


In [10]:
df["Department"].unique()

array(['Cardiology', 'Neurology', 'Orthopedics', 'Pediatrics',
       'Emergency', 'General Medicine', 'Oncology', 'Dermatology',
       'Gynecology', 'ENT', 'Gastroenterology', 'Pulmonology', 'Urology'],
      dtype=object)

In [11]:
df["Month"].unique()

array(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep',
       'Oct', 'Nov', 'Dec'], dtype=object)

In [12]:
(df["Patient_Volume"] < 0).sum()

np.int64(0)

In [13]:
(df["Average_Treatment_Duration"] <= 0).sum()

np.int64(0)

In [14]:
(df["Staff_Count"] <= 0).sum()

np.int64(0)

In [15]:
((df["Equipment_Utilisation"] < 0) |
 (df["Equipment_Utilisation"] > 100)).sum()

np.int64(0)

EDA


Department-wise patient volume

In [16]:
department_patients = df.groupby("Department")["Patient_Volume"].mean()

department_patients.sort_values(ascending=False)

,Patient_Volume
Department,
Emergency,1134.166667
General Medicine,958.500000
Cardiology,859.833333
Orthopedics,793.833333
Pediatrics,732.916667
Gynecology,643.416667
Neurology,596.666667
Pulmonology,583.416667
Gastroenterology,527.750000


Department-wise equipment utilisation

In [17]:
department_utilisation = df.groupby("Department")["Equipment_Utilisation"].mean()

department_utilisation.sort_values(ascending=False)

,Equipment_Utilisation
Department,
Emergency,97.200000
Cardiology,93.433333
General Medicine,79.333333
Pulmonology,74.475000
Oncology,74.391667
Neurology,73.508333
Orthopedics,72.333333
Gastroenterology,69.508333
Urology,69.250000


Department-wise staff

In [18]:
df.groupby("Department")["Staff_Count"].mean().sort_values(ascending=False)

,Staff_Count
Department,
General Medicine,25.000000
Emergency,23.916667
Pediatrics,21.833333
Orthopedics,20.333333
Gynecology,18.166667
Cardiology,17.666667
Neurology,16.000000
Pulmonology,15.666667
Gastroenterology,15.333333


Create Staff-to-Patient Ratio

In [19]:
df["Patients_Per_Staff"] = (
    df["Patient_Volume"] / df["Staff_Count"]
)

In [20]:
df[[
    "Department",
    "Patient_Volume",
    "Staff_Count",
    "Patients_Per_Staff"
]].head()

,Department,Patient_Volume,Staff_Count,Patients_Per_Staff
0,Cardiology,834,18,46.333333
1,Neurology,598,16,37.375000
2,Orthopedics,748,20,37.400000
3,Pediatrics,695,22,31.590909
4,Emergency,1051,24,43.791667


In [21]:
staff_ratio = (
    df.groupby("Department")["Patients_Per_Staff"]
      .mean()
      .sort_values(ascending=False)
)

staff_ratio

,Patients_Per_Staff
Department,
Cardiology,48.660475
Emergency,47.459463
Orthopedics,39.084127
General Medicine,38.342404
Urology,37.456105
Neurology,37.308885
Pulmonology,37.259722
Gynecology,35.467005
ENT,34.874145


Create a Resource Pressure Indicator

In [22]:
df["High_Equipment_Utilisation"] = (
    df["Equipment_Utilisation"] >= 80
).astype(int)

Seasonal Analysis

In [23]:
month_order = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

df["Month"] = pd.Categorical(
    df["Month"],
    categories=month_order,
    ordered=True
)

In [24]:
monthly_patients = (
    df.groupby("Month", observed=True)["Patient_Volume"]
      .sum()
)

monthly_patients

,Patient_Volume
Month,
Jan,8304
Feb,7984
Mar,8226
Apr,8946
May,9224
Jun,9082
Jul,8028
Aug,7725
Sep,7315


Identify consistently high-utilisation departments

In [25]:
utilisation_summary = (
    df.groupby("Department")["Equipment_Utilisation"]
      .agg(["mean", "min", "max", "std"])
      .sort_values("mean", ascending=False)
)

utilisation_summary

,mean,min,max,std
Department,,,,
Emergency,97.200000,93.5,98.0,1.594308
Cardiology,93.433333,87.5,96.4,2.795884
General Medicine,79.333333,72.8,83.7,3.343741
Pulmonology,74.475000,71.6,78.4,2.289551
Oncology,74.391667,69.4,80.6,3.888084
Neurology,73.508333,69.3,77.2,2.218705
Orthopedics,72.333333,66.6,80.9,3.697501
Gastroenterology,69.508333,65.3,75.5,3.551557
Urology,69.250000,64.2,73.2,3.353655


Statistical Analysis

In [26]:
from scipy.stats import f_oneway

In [27]:
groups = [
    group["Equipment_Utilisation"].values
    for name, group in df.groupby("Department")
]

In [28]:
statistic, p_value = f_oneway(*groups)

print("F-statistic:", statistic)
print("P-value:", p_value)

F-statistic: 145.3224667933435
P-value: 1.2645849217209593e-71


If p-value < 0.05:
    Significant difference exists between departments.

If p-value >= 0.05:
    No statistically significant difference detected.

Prepare Random Forest

In [29]:
X = df[
    [
        "Month",
        "Department",
        "Patient_Volume",
        "Average_Treatment_Duration",
        "Staff_Count"
    ]
]

y = df["High_Equipment_Utilisation"]

In [30]:
X = pd.get_dummies(
    X,
    columns=["Month", "Department"],
    drop_first=True
)

Train/Test Split

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [32]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

Make Predictions

In [33]:
y_pred = model.predict(X_test)

Evaluate the Model

In [34]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, zero_division=0))

Accuracy: 0.9
Precision: 1.0
Recall: 0.5
F1 Score: 0.6666666666666666


In [35]:
print(classification_report(y_test, y_pred, zero_division=0))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94        24
           1       1.00      0.50      0.67         6

    accuracy                           0.90        30
   macro avg       0.94      0.75      0.80        30
weighted avg       0.91      0.90      0.89        30



In [36]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[24  0]
 [ 3  3]]


In [37]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance.head(10)

,Feature,Importance
0,Patient_Volume,0.362779
2,Staff_Count,0.168121
1,Average_Treatment_Duration,0.167465
16,Department_Emergency,0.102750
18,Department_General Medicine,0.028635
22,Department_Orthopedics,0.022479
6,Month_May,0.017046
23,Department_Pediatrics,0.014984
8,Month_Jul,0.014972
20,Department_Neurology,0.011798


In [46]:
import os

output_path = "/content/drive/MyDrive/Hospital_Resource_Allocation/Results/processed_hospital_data.csv"

# Create the directory if it does not exist
output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)

df.to_csv(output_path, index=False)

In [50]:
import os

output_path_importance = "/content/drive/MyDrive/Hospital_Resource_Allocation/Results/feature_importance.csv"
output_dir_importance = os.path.dirname(output_path_importance)
os.makedirs(output_dir_importance, exist_ok=True)

importance.to_csv(
    output_path_importance,
    index=False
)